In [2]:
import pandas as pd

columnas_interes = [
    "CodigoExterno", "FechaPublicacion", "CodigoOrganismo", 
    "NombreOrganismo", "sector", "RegionUnidad", "Tipo de Adquisicion", 
    "TipoConvocatoria", "Estado", "CodigoProductoONU", "Rubro1", 
    "Rubro2", "Rubro3", "Cantidad", "CodigoMoneda", "MontoUnitarioOferta", 
    "CodigoProveedor", "Oferta seleccionada"
]

# 1. Cargar datos y eliminar nulos críticos
df = pd.read_csv(
    "data/df_completo.csv", 
    sep=";", 
    encoding="latin-1", 
    usecols=columnas_interes, 
    low_memory=False
)
df_limpio = df.dropna(subset=["sector", "FechaPublicacion"])

# 2. Deduplicar: Conservar 1 sola fila por licitación
df_unico = df_limpio.drop_duplicates(subset=["CodigoExterno"]).copy()

print(f"Total de licitaciones únicas: {len(df_unico)}")

Total de licitaciones únicas: 399410


In [3]:
df_unico['FechaPublicacion'] = pd.to_datetime(df_unico['FechaPublicacion'], errors='coerce')
df_unico = df_unico.dropna(subset=['FechaPublicacion']) # Eliminar fechas corruptas

# Crear columna de periodo semanal continuo (Lunes a Domingo)
df_unico['Semana'] = df_unico['FechaPublicacion'].dt.to_period('W')

# 4. Agrupar (Serie de tiempo): Contar licitaciones por Semana y Sector
serie_tiempo = df_unico.groupby(['Semana', 'sector']).size().reset_index(name='Volumen_Licitaciones')

# Ordenar cronológicamente
serie_tiempo = serie_tiempo.sort_values(by=['sector', 'Semana'])
print(serie_tiempo.head())

                   Semana sector  Volumen_Licitaciones
0   2023-01-02/2023-01-08   FFAA                   118
7   2023-01-09/2023-01-15   FFAA                   134
14  2023-01-16/2023-01-22   FFAA                   181
21  2023-01-23/2023-01-29   FFAA                   179
28  2023-01-30/2023-02-05   FFAA                   189


In [4]:
serie_tiempo.to_csv("data/serie_tiempo_limpia.csv", index=False)